# Manually annotating the resolvi model of the ULMS G4X dataset
- annotates the resolvi clustered object (proseg)
- This is for the revision of the paper

In [2]:
import os
import sys
import numpy as np
import scanpy as sc
import torch
import scvi
import pandas as pd
import anndata as ad
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpascvi

/home/jpagolia/miniforge3/envs/scvi-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# version control
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("scvi:", scvi.__version__)

plt.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.interactive = False
plt.ioff()
sc.settings.autoshow = False
plt.rcParams['savefig.dpi'] = 300
sc.set_figure_params(dpi_save=300, facecolor='white')

sc.settings.n_jobs = -1  # Use all available cores
scvi.settings.seed = 1234
torch.set_float32_matmul_precision("high")

[rank: 0] Seed set to 1234


pandas: 2.3.1
numpy: 2.2.6
scanpy: 1.11.4
scvi: 1.3.3


In [4]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
print(PARENT_DIR)

# Making an output directory using the pathlib package
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)

DATA_DIR = PARENT_DIR / 'resolvi'
print(f"DATA_DIR is {DATA_DIR}")

/oak/stanford/groups/longaker/ULMS/revision/G4X
Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/annotation
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/annotation
DATA_DIR is /oak/stanford/groups/longaker/ULMS/revision/G4X/resolvi


# Load the data

In [5]:
adata = sc.read_h5ad(DATA_DIR / 'resolvi_clustered_corr.h5ad')
adata

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', '_indices', '_scvi_batch', '_scvi_ind_x', '_scvi_labels', 'leiden0_1', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0', 'leiden1_1', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'true_proportion', 'diffusion_proportion', 'background_proportion'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors', '_scvi_manager_uuid', '_scvi_uuid', 'dendrogram_leiden0_1', 'dendrogram_leiden0_2', 'dendrogram_leiden0_3', 'dendrogram_leiden0_4', 'dendrogram_leiden0_5', 'dendrogram_leiden0_6', 'dendrogram_leiden0_7', 'dendrogram_leiden0_8', 'dendrogram_leiden0_9', 'dendrogram_leiden1_0', 'dendrogram_le

# Manual annotation of coarse cell types

In [ ]:
# resolution 0.4
leiden_key = 'leiden0_4'
leiden_map = {
    '0' : 'Tumor',
    '1' : 'Tumor',
    '2' : 'Vascular',
    '3' : 'Tumor',
    '4' : 'Stromal',
    '5' : 'Tumor',
    '6' : 'Tumor',
    '7' : 'Tumor',
}
adata.obs['coarse_celltype'] = adata.obs[leiden_key].map(leiden_map)
sc.pl.umap(adata, color='coarse_celltype', save='coarse_celltype.png')

In [ ]:
%matplotlib inline

In [ ]:
sc.pl.dotplot(adata, ["ESR1", "PGR", "AR"], groupby="leiden1_0", standard_scale='var')

In [ ]:
sc.pl.umap(adata, color='leiden1_0', legend_loc='on data')

In [ ]:
adata.write_h5ad('coarse_celltype.h5ad')

# Save individual annotated anndatas

In [7]:
adata = sc.read_h5ad('coarse_celltype.h5ad')
adata

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', '_indices', '_scvi_batch', '_scvi_ind_x', '_scvi_labels', 'leiden0_1', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0', 'leiden1_1', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'true_proportion', 'diffusion_proportion', 'background_proportion', 'coarse_celltype'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors', '_scvi_manager_uuid', '_scvi_uuid', 'coarse_celltype_colors', 'dendrogram_leiden0_1', 'dendrogram_leiden0_2', 'dendrogram_leiden0_3', 'dendrogram_leiden0_4', 'dendrogram_leiden0_5', 'dendrogram_leiden0_6', 'dendrogram_leiden0_7', 'dendrogram_leiden0_8', 'dendrogram_leide

In [9]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'ad')

Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/ad


In [10]:
sections = np.unique(adata.obs['Section'].tolist())
sections

array(['A02', 'A03', 'A04', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02',
       'C03', 'C04', 'D01', 'D03', 'D04', 'E01', 'E02', 'E03', 'F01',
       'F02', 'F03', 'G01', 'G02', 'G03', 'H01', 'H02', 'H03'],
      dtype='<U3')

In [14]:
output_dir

PosixPath('/oak/stanford/groups/longaker/ULMS/revision/G4X/ad')

In [15]:
for section in sections:
    section_adata = adata[adata.obs['Section'] == section]
    section_adata.write_h5ad(output_dir / f'{section}.h5ad')
    print(f'Wrote annotated anndata for section {section}')

Wrote annotated anndata for section A02
Wrote annotated anndata for section A03
Wrote annotated anndata for section A04
Wrote annotated anndata for section B01
Wrote annotated anndata for section B02
Wrote annotated anndata for section B03
Wrote annotated anndata for section B04
Wrote annotated anndata for section C01
Wrote annotated anndata for section C02
Wrote annotated anndata for section C03
Wrote annotated anndata for section C04
Wrote annotated anndata for section D01
Wrote annotated anndata for section D03
Wrote annotated anndata for section D04
Wrote annotated anndata for section E01
Wrote annotated anndata for section E02
Wrote annotated anndata for section E03
Wrote annotated anndata for section F01
Wrote annotated anndata for section F02
Wrote annotated anndata for section F03
Wrote annotated anndata for section G01
Wrote annotated anndata for section G02
Wrote annotated anndata for section G03
Wrote annotated anndata for section H01
Wrote annotated anndata for section H02
